In [ ]:
!pip install google-genai langchain faiss-cpu python-dotenv
!pip install langchain[all]
!pip install sentence-transformers
!pip install datasets huggingface-hub
!pip install litellm
!pip install -U pypdf
!pip install langchain-core
!pip install langchain-huggingface
!pip install langchain-google-genai
!pip install langchain-text-splitters
!pip install faiss-cpu
!pip install -U langchain-community

In [ ]:
import langchain
print(langchain.__version__)
print(langchain)
print(dir(langchain))
print(langchain.__package__)

1.3.1
<module 'langchain' from '/usr/local/lib/python3.12/dist-packages/langchain/__init__.py'>
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__']
langchain


In [ ]:
import requests as rq
import bs4
import json
from dotenv import load_dotenv
import langchain as lc
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
from sentence_transformers import SentenceTransformer, util
import torch
import warnings
warnings.filterwarnings("ignore")
import logging

logging.getLogger("pypdf").setLevel(logging.ERROR)





In [ ]:
import requests as rq
from bs4 import BeautifulSoup
import json

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

def _search_google(search):
    resp = rq.get(
        "https://www.google.com/search",
        params={"q": search, "hl": "en"},
        headers=HEADERS,
        timeout=10,
    )
    soup = BeautifulSoup(resp.text, "html.parser")
    results = []

    for item in soup.select("div.g")[:5]:
        title_tag = item.select_one("h3")
        link_tag = item.select_one("a")
        snippet_tag = item.select_one(".VwiC3b, .IsZvec")

        if title_tag and link_tag:
            results.append({
                "title": title_tag.get_text(strip=True),
                "link": link_tag.get("href"),
                "snippet": snippet_tag.get_text(strip=True) if snippet_tag else "",
            })
    return results

def _search_duckduckgo(search):
    resp = rq.get(
        "https://html.duckduckgo.com/html",
        params={"q": search},
        headers=HEADERS,
        timeout=10,
    )
    soup = BeautifulSoup(resp.text, "html.parser")
    results = []

    for item in soup.select(".result"):
        title_tag = item.select_one("a.result__a")
        snippet_tag = item.select_one("a.result__snippet, .result__snippet")
        if title_tag:
            results.append({
                "title": title_tag.get_text(strip=True),
                "link": title_tag.get("href"),
                "snippet": snippet_tag.get_text(strip=True) if snippet_tag else "",
            })
    return results

def _search_bing(search):
    resp = rq.get(
        "https://www.bing.com/search",
        params={"q": search},
        headers=HEADERS,
        timeout=10,
    )
    soup = BeautifulSoup(resp.text, "html.parser")
    results = []

    for item in soup.select("li.b_algo"):
        title_tag = item.select_one("h2 a")
        snippet_tag = item.select_one("p")
        if title_tag:
            results.append({
                "title": title_tag.get_text(strip=True),
                "link": title_tag.get("href"),
                "snippet": snippet_tag.get_text(strip=True) if snippet_tag else "",
            })
    return results

def _search_ecosia(search):
    resp = rq.get(
        "https://www.ecosia.org/search",
        params={"q": search},
        headers=HEADERS,
        timeout=10,
    )
    soup = BeautifulSoup(resp.text, "html.parser")
    results = []

    for item in soup.select(".result"):
        title_tag = item.select_one("h3 a")
        snippet_tag = item.select_one(".snippet")
        if title_tag:
            results.append({
                "title": title_tag.get_text(strip=True),
                "link": title_tag.get("href"),
                "snippet": snippet_tag.get_text(strip=True) if snippet_tag else "",
            })
    return results

def search_engine(search):
    all_results = []
    seen_links = set()
    engines = [
        ("google", _search_google),
        ("duckduckgo", _search_duckduckgo),
        ("bing", _search_bing),
        ("ecosia", _search_ecosia),
    ]

    for engine_name, search_fn in engines:
        try:
            results = search_fn(search)
            for result in results:
                link = result.get("link")
                if link and link not in seen_links:
                    seen_links.add(link)
                    result["source"] = engine_name
                    all_results.append(result)
        except Exception as e:
            all_results.append({
                "source": engine_name,
                "error": str(e),
            })

    return json.dumps({
        "search": search,
        "results": all_results
    }, ensure_ascii=False)

print(search_engine("nice"))

# rese=web_crawl()
# print(rese)

{"search": "nice", "results": [{"title": "The 10 Best Hotels in Nice - Nice In France", "link": "//duckduckgo.com/l/?uddg=https%3A%2F%2Fduckduckgo.com%2Fy.js%3Fad_domain%3Dtripadvisor.com.sg%26ad_provider%3Dbingv7aa%26ad_type%3Dtxad%26click_metadata%3DC9VKfPb6CcWlbBmdKVpOE_8t11bq1EguApLep8dYjmM8pFL5MVaq3dCTubwk5qMi5buXGVsRsM80JPDhJikbQ65Ps_xISmZk%252DQkjevMPSh0oyNk4DDNubwhivJwGjNvPCEdmDgcRQsH2NHQyc8W1N0rGVP%252Da5SUVCrLIgfkkqnk.qkkhLM8fjqpnKExpHueu%252Dg%26rut%3D4ed6d3b797998efcd047a32d042cfbea803f6fcf9b6dab9bc7c161f117eff816%26u3%3Dhttps%253A%252F%252Fwww.bing.com%252Faclick%253Fld%253De8ua_KqqRLC1WKRHEYqUUjszVUCUzIIWssEKlUBsL4gEL8jXLJyycyAmG7ScQciwyNWWXuCiExpWpKipiPdpQYlkAewwOLhRKisohuKLhG0t8kQbF0LwstOHnhrTLsxGhIczk2JDk6Z%252DtfxVyLSKjKgsy8NTRBAxSSUwSV%252DQacljZ4YiFqgHl%252DNd5bqJFw8Jn4Knhp0MXYKf_NFqEXtdxv6wS413w%2526u%253DaHR0cHMlM2ElMmYlMmZ3d3cudHJpcGFkdmlzb3IuY29tLnNnJTJmU21hcnREZWFscyUzZmdlbyUzZDE4NzIzNCUyNm0lM2QxODIwNCUyNnN1cGNtJTNkMzMzMTY1OTcxJTI2c3VwYWclM2QxMjE5MzU4NTQzNTc5MD

In [ ]:
import requests as rq
import json

SEARCHAPI_KEY = " "

def search_engine_api(query):
    """Use SearchAPI.io for search results"""
    try:
        response = rq.get(
            "https://www.searchapi.io/api/v1/search",
            params={
                "q": query,
                "engine": "google",
            },
            headers={"Authorization": f"Bearer {SEARCHAPI_KEY}"},
            timeout=10,
        )
        response.raise_for_status()

        data = response.json()
        results = []

        for item in data.get("organic_results", [])[:5]:
            results.append({
                "title": item.get("title", ""),
                "link": item.get("link", ""),
                "snippet": item.get("snippet", ""),
            })

        return json.dumps({
            "search": query,
            "results": results
        }, ensure_ascii=False)

    except Exception as e:
        print(f"SearchAPI error: {e}")
        return json.dumps({
            "search": query,
            "results": [],
            "error": str(e)
        }, ensure_ascii=False)

# Test
print(search_engine("diabetes"))

{"search": "diabetes", "results": [{"title": "Check for Diabetes - Healthier SG Screening", "link": "//duckduckgo.com/l/?uddg=https%3A%2F%2Fduckduckgo.com%2Fy.js%3Fad_domain%3Dhealth.gov.sg%26ad_provider%3Dbingv7aa%26ad_type%3Dtxad%26click_metadata%3DgOWRiVzIbwHTaO3n__h412WxG5FXOlv180SePjZqnl6fshGrquUYXT8lcVLiXuzdWLmAKe5c8PqDo4c_j%252Df_csUP878o3MpVok2i4xFZs0oPR50QOo57S9PI0fMRG2isBfvNvFLkkixvayQg1BIc7wDnlhpqbCUwlt6TJWJu9XM.9vzz2_3_B3tXAMv4MermdA%26rut%3D6ae0f34a8b02fd64f4466af12340851a197ca3e3639f1b39a52ef640c376653e%26u3%3Dhttps%253A%252F%252Fwww.bing.com%252Faclick%253Fld%253De8twib1u5tpIh9qSmufEtiHjVUCUya1RTlnoBeoHsh12qsi7tVVacT2BFCeQimqfgSPTXbuLcGe2dk2O9XeHVE%252DNHRCqZq1htaQjYw44NMJKbBi7y%252Drhv_cwVlkfl1dlfVB7sK%252DJun28BoQYjiScNKy9tcOROszKuBiVZYr9Mqm9gc7J36ujIn3jiOkbgSYHrJVExhbRrtKd4srl6RbBFC3x4yD5Y%2526u%253DaHR0cHMlM2ElMmYlMmZib29rLmhlYWx0aC5nb3Yuc2clMmZoZWFsdGhpZXJzZy1zY3JlZW5pbmclM2Z1dG1fc291cmNlJTNkYmluZyUyNnV0bV9tZWRpdW0lM2RwYWlkLXNlYXJjaCUyNnV0bV9jYW1wYWlnbiUzZGZ5MjYtaHN

In [ ]:
hardcoded =[
    "https://bookchapter.org/kitaplar/Medical%20Diagnosis%20and%20Treatment%20Methods%20in%20Basic%20Medical%20Sciences.pdf",
    "https://applications.emro.who.int/dsaf/dsa236.pdf",
    "https://applications.emro.who.int/dsaf/dsa509.pdf",
    "https://applications.emro.who.int/dsaf/dsa664.pdf",
   "https://www.icmr.gov.in/icmrobject/custom_data/pdf/resource-guidelines/ICMR_GuidelinesType2diabetes2018_0.pdf",
    "https://rlmc.edu.pk/themes/images/gallery/library/books/Microbiology/Text_Book_of_Microbiology.pdf", #Microbiology
    "https://ndl.ethernet.edu.et/bitstream/123456789/14266/1/1410.pdf",# Clinical Methods
    "https://www.sssutms.co.in/cms/Areas/Website/Files/Link/EContent/ANATOMY_PHYSIOLOGY.pdf", #Anatomy & Physiology
    "https://rlmc.edu.pk/themes/images/gallery/library/books/Pharmacology/GENERAL%20PRINCIPLES%20OF%20PHARMACOLOGY.pdf",#Pharmacology
    "https://sist.sathyabama.ac.in/sist_coursematerial/uploads/SMB1102.pdf",#Biochemistry
   "https://awesomechem.files.wordpress.com/2016/10/harpers-illustrated-biochemistry-28th-ed-robert-k-murray-et-al-mcgraw-hill-2009.pdf",#Biochemistry
"https://www.entuk.org/_userfiles/pages/files/groups/ear_nose_and_throat_the_offici_sfo_uk_compressed.pdf",
"https://thelifesaversblog.wordpress.com/wp-content/uploads/2015/11/r-jogi-basic-ophthalmology-4th-edition.pdf",
    "https://www.digital.avicennamch.com/updata/services/file_file/364_20190306185233.pdf", #gynocology
    "https://aiimsrishikesh.edu.in/newwebsite/wp-content/uploads/2018/09/782_Principles_of_Radiology.pdf"

    ]

In [ ]:
import os
import re
import logging
import tempfile
import subprocess
import requests as rq
from bs4 import BeautifulSoup
from langchain_community.document_loaders import PyPDFLoader

# Suppress pypdf warnings
logging.getLogger('pypdf').setLevel(logging.ERROR)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/137.0.0.0 Safari/537.36"
    ),
    "Accept": "application/pdf,application/octet-stream,*/*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/",
    "Connection": "keep-alive",
}

def get_ncbi_pdf_url(book_id):
    """Extract PDF from NCBI book page"""
    try:
        url = f"https://www.ncbi.nlm.nih.gov/books/{book_id}/"
        r = rq.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(r.text, 'html.parser')
        pdf_link = soup.find('a', {'data-ga-action': 'PDF'})
        if pdf_link and pdf_link.get('href'):
            return "https://www.ncbi.nlm.nih.gov" + pdf_link['href']
    except Exception as e:
        print(f"NCBI extract failed: {e}")
    return None

def _download_with_curl(url):
    with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp:
        cmd = [
            "curl", "-L", "-A", HEADERS["User-Agent"],
            "-e", HEADERS["Referer"],
            "--retry", "3", "--retry-delay", "2",
            "-o", tmp.name, url,
        ]
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        if result.returncode != 0:
            raise RuntimeError(result.stderr)
        return tmp.name

def load_pdf(url):
    if not url.startswith(("http://", "https://")):
        raise ValueError(f"Invalid URL: {url}")

    # Handle NCBI books
    if "ncbi.nlm.nih.gov/books/" in url:
        match = re.search(r'/books/(NBK\d+)', url)
        if match:
            book_id = match.group(1)
            pdf_url = get_ncbi_pdf_url(book_id)
            if pdf_url:
                url = pdf_url
                print(f"Using extracted NCBI PDF: {url}")
            else:
                return None

    local_path = None
    try:
        session = rq.Session()
        response = session.get(
            url, headers=HEADERS, allow_redirects=True,
            timeout=60, stream=True,
        )

        print(f"url: {url}")
        print(f"Status: {response.status_code}")
        print(f"Reason: {response.reason}")
        print(f"Content-Type: {response.headers.get('Content-Type')}")

        # Check for error pages
        content_type = response.headers.get('Content-Type', '').lower()
        if 'xml' in content_type or 'html' in content_type:
            print("❌ Got error page instead of PDF")
            return None

        response.raise_for_status()

        with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp:
            for chunk in response.iter_content(8192):
                if chunk:
                    tmp.write(chunk)
            local_path = tmp.name

    except Exception as e:
        print(f"Requests failed: {e}")
        print("Trying curl fallback...")
        local_path = _download_with_curl(url)

    if os.path.getsize(local_path) == 0:
        raise ValueError("Downloaded file is empty")

    try:
        loader = PyPDFLoader(local_path)
        docs = loader.load()
        return docs
    finally:
        try:
            os.remove(local_path)
        except:
            pass

# Usage
docs = load_pdf(hardcoded[0])
if docs:
    print(f"✓ Pages: {len(docs)}")
    print(docs[0].page_content[:500])
else:
    print("Failed to load PDF")

url: https://bookchapter.org/kitaplar/Medical%20Diagnosis%20and%20Treatment%20Methods%20in%20Basic%20Medical%20Sciences.pdf
Status: 200
Reason: OK
Content-Type: application/pdf
✓ Pages: 147
Editor
Prof. Dr. Sıddık Keskin
Medical Diagnosis
and
Treatment Methods
in
Basic Medical Sciences
Medical Diagnosis and Treatment Methods in Basic Medical Sciences
Health
livredelyon.com
livredelyon
livredelyon
livredelyon
ISBN: 978-2-38236-061-3


In [ ]:
from urllib.parse import urlparse
import requests as rq
from bs4 import BeautifulSoup
import json

all_documents = []
queries = [
    "site:who.int diabetes",
    "site:who.int hypertension",
    "site:who.int tuberculosis",
    "site:cdc.gov asthma",
    "site:nih.gov cancer",
    "site:medlineplus.gov pneumonia",
    "site:ncbi.nlm.nih.gov stroke",
    "site:who.int malaria",
    "site:who.int dengue",
    "site:nih.gov epilepsy",
    "site:medlineplus.gov heart disease",
    "site:cdc.gov influenza",
    "site:ncbi.nlm.nih.gov microbiology",
    "site:ncbi.nlm.nih.gov pharmacology",
]

ALLOWED_DOMAINS = {
    "who.int",
    "nih.gov",
    "ncbi.nlm.nih.gov",
    "medlineplus.gov",
    "mayoclinic.org",
    "msdmanuals.com",
    "cdc.gov",
}

def web_crawl():
    crawled_count = 0
    added_count = 0
    failed_count = 0

    for query in queries:
        print(f"\n🔍 Searching: {query}")

        try:
            response_json = search_engine_api(query)  # Now works!
            search_results = json.loads(response_json).get("results", [])

            if not search_results:
                print(f"  No results found")
                continue

        except Exception as e:
            print(f"Search failed: {e}")
            continue

        for result in search_results:
            url = result["link"]
            crawled_count += 1

            try:
                domain = urlparse(url).netloc

                if not any(allowed in domain for allowed in ALLOWED_DOMAINS):
                    continue

                response = rq.get(url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
                response.raise_for_status()

                soup = BeautifulSoup(response.text, "html.parser")
                text = soup.get_text(" ", strip=True)

                if len(text) < 500:
                    continue

                all_documents.append({
                    "query": query,
                    "url": url,
                    "title": result["title"],
                    "content": text
                })

                added_count += 1
                print(f"✓ Added ({added_count}): {url[:60]}...")

            except Exception as e:
                failed_count += 1

    print(f"\n{'='*50}\n📊 Total crawled: {crawled_count} | Added: {added_count}\n{'='*50}\n")
    return all_documents


# docs = web_crawl()
# print(f"Documents collected: {len(docs)}")

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "qiaojin/PubMedQA",
    "pqa_labeled",
    split="train"
)


pubmed_documents = []

for row in dataset:
    context = " ".join(row["context"]["contexts"])

    pubmed_documents.append(
        {
            "content": context
        }
    )

In [ ]:
from langchain_core.documents import Document

documents = []

# PDFs
valid_hardcoded = [url for url in hardcoded if url and url.strip()]
for url in valid_hardcoded:
    try:
        docs = load_pdf(url)
        documents.extend(docs)



    except Exception as exc:
        print(exc)

documents.extend(pubmed_documents)

# Websites
data = web_crawl()

for item in data:
    documents.append(
        Document(
            page_content=item["content"],
            metadata={
                "source": item["url"]
            }
        )
    )
#print number of websites,pdfs,dataset collected
print("Number of websites:", len(data))
print("Number of PDFs:", len(documents))
print("Number of documents:", len(documents))

url: https://bookchapter.org/kitaplar/Medical%20Diagnosis%20and%20Treatment%20Methods%20in%20Basic%20Medical%20Sciences.pdf
Status: 200
Reason: OK
Content-Type: application/pdf
url: https://applications.emro.who.int/dsaf/dsa236.pdf
Status: 200
Reason: OK
Content-Type: application/pdf
url: https://applications.emro.who.int/dsaf/dsa509.pdf
Status: 200
Reason: OK
Content-Type: application/pdf
url: https://applications.emro.who.int/dsaf/dsa664.pdf
Status: 200
Reason: OK
Content-Type: application/pdf
url: https://www.icmr.gov.in/icmrobject/custom_data/pdf/resource-guidelines/ICMR_GuidelinesType2diabetes2018_0.pdf
Status: 200
Reason: OK
Content-Type: application/pdf
url: https://rlmc.edu.pk/themes/images/gallery/library/books/Microbiology/Text_Book_of_Microbiology.pdf
Status: 200
Reason: OK
Content-Type: application/pdf
Requests failed: HTTPSConnectionPool(host='ndl.ethernet.edu.et', port=443): Max retries exceeded with url: /bitstream/123456789/14266/1/1410.pdf (Caused by SSLError(SSLCertVe

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, util
from langchain_core.documents import Document

# Split documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# Check if documents were loaded
if not documents:
    raise ValueError("No documents were loaded. Check the PDF loading step.")

# Convert pubmed_documents (which are dicts) to Document objects
processed_documents = []
for doc in documents:
    if isinstance(doc, dict) and 'content' in doc:
        processed_documents.append(Document(page_content=doc['content']))
    elif hasattr(doc, 'page_content'):
        processed_documents.append(doc)
    else:
        # Skip or handle other types if necessary
        continue

chunks = text_splitter.split_documents(processed_documents)

model = SentenceTransformer(
    "BAAI/bge-large-en-v1.5"
)

texts = [doc.page_content for doc in chunks]

doc_embeddings = model.encode(
    texts,
    convert_to_tensor=True,
    show_progress_bar=True
)

query = "What are the common diagnostic methods for diabetes?"

query_embedding = model.encode(
    query,
    convert_to_tensor=True
)

scores = util.cos_sim(
    query_embedding,
    doc_embeddings
)[0]

top_k = min(5, len(texts))

top_results = torch.topk(scores, k=top_k)

retrieved_docs = [
    texts[idx.item()]
    for idx in top_results.indices
]

context = "\n\n".join(retrieved_docs)

print("--- Retrieved Context (First 2000 chars) ---")
print(context[:2000])
print("\n--- Statistics ---")
print("len(chunks):", len(chunks))
print("len(texts):", len(texts))
print("doc_embeddings shape:", doc_embeddings.shape)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/535 [00:00<?, ?it/s]

--- Retrieved Context (First 2000 chars) ---
30 Guidelines for the prevention, management and care of diabetes mellitus
Screening tools
Glucose measurement [21]
At present there is no satisfactory substitute to glucose measurement. Alternatives, 
such as measurements of glycated haemoglobin, glycated proteins and 1,5-anhydroglucitol, 
although specific, are too insensitive to reliably detect lesser degrees of glycaemic 
disturbances. There are many methods available for measuring blood glucose, ranging 
from visually-read test-strips to sophisticated automated methods. Precision and accuracy 
are required for screening. If portable meters are to be used, they should be checked under 
a full quality assurance programme and a coefficient of variation >5% should not be 
accepted. When automated procedures are used, care must be taken to minimize the risk 
of errors in sample identification.
Oral glucose tolerance test (OGGT)
The oral glucose tolerance test remains the definitive confirmat

In [ ]:
from google.colab import drive
import pickle

# 1. Mount your personal Google Drive
# This will trigger a popup asking you to log in and grant access
drive.mount('/content/drive')

# 2. Define the path where you want to save the file in your Drive
# /content/drive/MyDrive/ is the root folder of your Google Drive
drive_path = '/content/drive/MyDrive/www/rag_store.pkl'

with open(drive_path, "wb") as f:
    pickle.dump(
        {
            "embeddings": doc_embeddings,
            "texts": texts
        },
        f
    )

print(f"Successfully saved to {drive_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Successfully saved to /content/drive/MyDrive/www/rag_store.pkl


In [ ]:
# doc_embeddings

In [ ]:
def generate(query):
    result = qa_chain(query)
    print("Answer:", result["result"])
    print("\nSources:")
    for doc in result["source_documents"]:
        print(f"- {doc.metadata['source']}")
    return result

In [ ]:
userquery = "What are the symptoms of diabetes?"


In [ ]:
# Hugging Face collections are not directly loadable with datasets.load_dataset.
# Use a specific dataset repo ID or download files from huggingface_hub instead.
print(type(model))

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


In [ ]:
print(len(documents))
print(len(chunks))
print(len(texts))
print(doc_embeddings.shape)
if len(texts) <= 0 or doc_embeddings.shape[0] <= 0 or len(chunks) <= 0:
    raise ValueError("No text chunks were created. Check the document loading and splitting steps.")



5240
17091
17091
torch.Size([17091, 1024])


In [ ]:
from datasets import load_dataset
import torch

def evaluate_recall(dataset_name="openlifescienceai/medmcqa", split="train", num_samples=1000, k=5):
    # Ensure we are using the globally defined model and embeddings
    global model, doc_embeddings, texts

    # Load dataset
    print(f"Loading dataset: {dataset_name}...")
    dataset = load_dataset(dataset_name, split=split)
    test_set = dataset.select(range(min(num_samples, len(dataset))))

    hits = 0

    print(f"Starting evaluation on {len(test_set)} samples using k={k}...")

    for i, item in enumerate(test_set):
        question = item["question"]

        # Retrieve top-k chunks
        query_embedding = model.encode(
            question,
            convert_to_tensor=True
        )

        scores = util.cos_sim(
            query_embedding,
            doc_embeddings
        )[0]

        top_results = torch.topk(
            scores,
            k=k
        )

        retrieved_docs = [
            texts[idx.item()]
            for idx in top_results.indices
        ]

        context = " ".join(retrieved_docs).lower()

        # Correct answer logic for MedMCQA (cop: 0=a, 1=b, 2=c, 3=d)
        answer_idx = item["cop"]
        options = [
            str(item["opa"]),
            str(item["opb"]),
            str(item["opc"]),
            str(item["opd"])
        ]

        correct_answer = options[answer_idx].lower()

        if correct_answer in context:
            hits += 1

        if (i + 1) % 100 == 0:
            print(f"Processed {i+1}/{len(test_set)} | Current Recall@{k}: {hits/(i+1):.4f}")

    final_recall = hits / len(test_set)
    print(f"\nFinal Results for {dataset_name}:")
    print(f"Recall@{k}: {final_recall:.4f}")
    return final_recall

# Run the evaluation with the current global state
if 'doc_embeddings' in globals() and 'texts' in globals():
    evaluate_recall(num_samples=500, k=5)
else:
    print("Error: doc_embeddings or texts not found. Please run the embedding cell first.")

Loading dataset: openlifescienceai/medmcqa...


README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/85.9M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/936k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.48M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/182822 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6150 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4183 [00:00<?, ? examples/s]

Starting evaluation on 500 samples using k=5...
Processed 100/500 | Current Recall@5: 0.1100
Processed 200/500 | Current Recall@5: 0.1150
Processed 300/500 | Current Recall@5: 0.1333
Processed 400/500 | Current Recall@5: 0.1500
Processed 500/500 | Current Recall@5: 0.1560

Final Results for openlifescienceai/medmcqa:
Recall@5: 0.1560


# generation part

In [ ]:
!pip install -q litellm google-genai

import os
from google import genai
from litellm import completion

# Set your API key
os.environ["GEMINI_API_KEY"] = ""

# Get all available Gemini text models
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

MODELS = []
try:
    for model_info in client.models.list():
        name = model_info.name.split("/")[-1]
        if "gemini" in name.lower() and "embed" not in name.lower():
            MODELS.append(f"gemini/{name}")
    MODELS = sorted(list(set(MODELS)))
    print("Loaded Models:", MODELS)
except Exception as e:
    print(f"Error loading models: {e}")
    MODELS = ["gemini/gemini-pro"]

current_model = 0

from litellm import completion
import time

current_model = 0

def ask_llm(
    prompt,
    temperature=0.2,
    max_tokens=2048,
    retries=2
):
    global current_model

    valid_models = [
        m for m in MODELS
        if (
            "embed" not in m.lower()
            and "image" not in m.lower()
            and "vision" not in m.lower()
            and "tts" not in m.lower()
            and "audio" not in m.lower()
        )
    ]

    if not valid_models:
        raise ValueError(
            "No valid text models found."
        )

    for _ in range(len(valid_models)):

        model_name = valid_models[current_model]

        current_model = (
            current_model + 1
        ) % len(valid_models)

        for attempt in range(retries):

            try:

                print(
                    f"\nUsing: {model_name}"
                )

                response = completion(
                    model=model_name,
                    messages=[
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ],
                    temperature=temperature,
                    max_tokens=max_tokens
                )

                return (
                    response
                    .choices[0]
                    .message
                    .content
                )

            except Exception as e:

                print(
                    f"{model_name} failed "
                    f"(attempt {attempt+1}): "
                    f"{str(e)[:200]}"
                )

                time.sleep(2)

                continue

    return (
        "All Gemini models failed. "
        "Check API quota."
    )

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 18.8 MB/s eta 0:00:00
Loaded Models: ['gemini/gemini-2.0-flash', 'gemini/gemini-2.0-flash-001', 'gemini/gemini-2.0-flash-lite', 'gemini/gemini-2.0-flash-lite-001', 'gemini/gemini-2.5-computer-use-preview-10-2025', 'gemini/gemini-2.5-flash', 'gemini/gemini-2.5-flash-image', 'gemini/gemini-2.5-flash-lite', 'gemini/gemini-2.5-flash-native-audio-latest', 'gemini/gemini-2.5-flash-native-audio-preview-09-2025', 'gemini/gemini-2.5-flash-native-audio-preview-12-2025', 'gemini/gemini-2.5-flash-preview-tts', 'gemini/gemini-2.5-pro', 'gemini/gemini-2.5-pro-preview-tts', 'gemini/gemini-3-flash-preview', 'gemini/gemini-3-pro-image', 'gemini/gemini-3-pro-image-preview', 'gemini/gemini-3-pro-preview', 'gemini/gemini-3.1-flash-image', 'gemini/gemini-3.1-flash-image-preview', 'gemini/gemini-3.1-flash-lite', 'gemini/gemini-3.1-flash-lite-preview', 'gemini/gemini-3.1-f

In [ ]:
from google.colab import drive
import os

print("Refreshing Drive connection...")
# This will prompt for authorization again to ensure a fresh connection
drive.mount('/content/drive', force_remount=True)

target_file = '/content/drive/MyDrive/www/rag_store.pkl'

if os.path.exists(target_file):
    print(f"\n✅ Success! File found at: {target_file}")
    print(f"File size: {os.path.getsize(target_file) / 1024:.2f} KB")
else:
    print(f"\n❌ Still cannot find the file at {target_file}")
    # List the folder to see what IS there
    parent_dir = os.path.dirname(target_file)
    if os.path.exists(parent_dir):
        print(f"Contents of {parent_dir}:", os.listdir(parent_dir))
    else:
        print(f"The directory {parent_dir} does not exist in Drive.")

Refreshing Drive connection...
Mounted at /content/drive

✅ Success! File found at: /content/drive/MyDrive/www/rag_store.pkl
File size: 82809.91 KB


In [ ]:
import pickle
import torch
from sentence_transformers import SentenceTransformer, util

drive_path = '/content/drive/MyDrive/www/rag_store.pkl'

with open(drive_path, "rb") as f:
    data = pickle.load(f)

embeddings = data["embeddings"]
texts = data["texts"]

if not torch.is_tensor(embeddings):
    embeddings = torch.tensor(embeddings)

model = SentenceTransformer(
    "BAAI/bge-large-en-v1.5"
)

query = "what is diabetes"

query_embedding = model.encode(
    query,
    convert_to_tensor=True
)

scores = util.cos_sim(
    query_embedding,
    embeddings
)[0]

top_k = 3

top_results = torch.topk(
    scores,
    k=min(top_k, len(texts))
)

retrieved_context = []

print("\n--- Retrieval Results ---")

for idx, score in zip(
    top_results.indices,
    top_results.values
):
    doc = texts[idx.item()]

    retrieved_context.append(doc)

    print(
        f"\nScore: {score.item():.4f}"
    )
    print(doc[:500])

context = "\n\n".join(
    retrieved_context
)

print("\nContext length:", len(context))





RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [ ]:
%%bash
# Install missing dependency zstd
sudo apt-get update && sudo apt-get install -y zstd

# Install the Ollama binary
curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in the background
nohup ollama serve > ollama.log 2>&1 &

In [ ]:
# Using absolute path to ensure the command is found immediately after installation
import time
import os

# Short delay to ensure the background service 'ollama serve' is ready
time.sleep(5)

print("--- Pulling Qwen 2.5 Model ---")
!/usr/local/bin/ollama pull qwen2.5:7b

print("\n--- Testing Model ---")
!/usr/local/bin/ollama run qwen2.5:7b "Hello, are you running on Colab?"

In [ ]:
print("\n--- Running Qwen 2.5 via Ollama ---")
# Construct the prompt as a Python string first to handle escaping and formatting
full_prompt = f"Context: {context} \n\nQuestion: {query} \n\nAnswer only from the context."

# Pass the Python variable 'full_prompt' to the shell command using {variable_name}
# We wrap it in quotes to handle spaces and newlines correctly in the terminal
!/usr/local/bin/ollama run qwen2.5:7b "{full_prompt}"

In [ ]:
from huggingface_hub import HfApi
import os

# Configuration
repo_id = "chintu4/med-bot"
local_file_path = "/content/drive/MyDrive/www/rag_store.pkl"
path_in_repo = "rag_store.pkl"
token = ""# Ensure you have added this to Colab Secrets or Env

api = HfApi()

try:
    print(f"Attempting to upload {local_file_path} to HF Space: {repo_id}...")

    api.upload_file(
        path_or_fileobj=local_file_path,
        path_in_repo=path_in_repo,
        repo_id=repo_id,
        repo_type="space",
        token=token
    )

    print("\n✅ Successfully pushed rag_store.pkl to Hugging Face!")
except Exception as e:
    print(f"\n❌ Error uploading to Hugging Face: {e}")
    print("Tip: Make sure your HUGGINGFACE_TOKEN has 'write' permissions and the repository path is correct.")

Attempting to upload /content/drive/MyDrive/www/rag_store.pkl to HF Space: chintu4/med-bot...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...MyDrive/www/rag_store.pkl:  40%|####      | 34.0MB / 84.8MB            


✅ Successfully pushed rag_store.pkl to Hugging Face!


In [ ]:
import os
import shutil

# --- Configuration ---
# Ensure this token has 'repo' (classic) or 'Contents: Write' (fine-grained) permissions
GITHUB_TOKEN = ""
GITHUB_USERNAME = "chintu4"
REPO_NAME = "medbot"
FILE_TO_UPLOAD = "/content/drive/MyDrive/www/rag_store.pkl"

if not os.path.exists(FILE_TO_UPLOAD):
    print(f"❌ Error: {FILE_TO_UPLOAD} not found. Check if Drive is mounted correctly.")
else:
    print(f"⌛ Initializing upload for {REPO_NAME}...")

    # Git Config
    !git config --global user.email "chintusharma00014@gmail.com"
    !git config --global user.name "chintu4"

    # Prepare clean workspace
    %cd /content
    !rm -rf github_upload
    !mkdir -p github_upload
    %cd github_upload
    !git init -q

    # Prepare file
    shutil.copy(FILE_TO_UPLOAD, "rag_store.pkl")
    !git add rag_store.pkl
    !git commit -m "Update RAG store from Colab" -q

    # Authenticated Push
    remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
    !git remote add origin {remote_url}
    !git branch -M main

    print("↑ Attempting push to GitHub...")
    # Use a variable to capture output for better debugging
    result = !git push -u origin main --force 2>&1
    output = "\n".join(result)

    if "denied" in output.lower() or "403" in output:
        print("❌ Access Denied (403). Please update your GitHub Token permissions to include 'Contents: Write'.")
        print(f"Details: {output}")
    elif "error" in output.lower() or "fatal" in output.lower():
        print(f"❌ Git Error: {output}")
    else:
        print(f"\n✅ Success! Data uploaded to: https://github.com/{GITHUB_USERNAME}/{REPO_NAME}")

⌛ Initializing upload for medbot...
/content
/content/github_upload
↑ Attempting push to GitHub...

✅ Success! Data uploaded to: https://github.com/chintu4/medbot


In [ ]:
embeddings = docs_embeddings.cpu()

store = {
    "embeddings": embeddings,
    "texts": texts
}

torch.save(store, "rag_store.pkl")

NameError: name 'docs_embeddings' is not defined

In [ ]:
import torch

store = torch.load(
    "/content/drive/MyDrive/www/rag_store.pkl",
    map_location="cpu",
    weights_only=False
)

print(type(store))

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.